# Data Preprocessing Tasks
**Jennifer Eigo - University of Connecticut - Dept. of Operations and Information Management**

-------------------------------------
In this module we will practice different techniques to modify the data to clean it up and get it ready for modeling.



# Environment Setup

In [ ]:
# import modules

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

!pip install scikit-learn
from sklearn.preprocessing import PowerTransformer

# Set display options to show all columns
pd.set_option('display.max_columns', None)


In [ ]:
# mount your google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# read data
# on the lefthand side, navigate to your data and copy the path
df = pd.read_csv('/content/drive/MyDrive/OPIM 5604 Python/Module 2/ToyotaCorolla.csv')

In [ ]:
# shape
# show how many rows and columns
# this file has 1436 rows and 38 columns
df.shape

In [ ]:
# Preview
print(df.head())

# Change Data Types

Let's look at the variable datatypes.  Wherever you see "object" it is a categorical variable.  Wherever you see "int64" it is a whole number.  Wherever you see "float64" it is a decimal number.  Some of this repeats steps we took in the last module, but we need to get the data types correct before continuing on with our work.  

In [ ]:
# data types
# look at the data types of the columns
df.dtypes

Looks like we have a lot of 0 and 1 binary indicators that are coded as continuous.  So let's check the data to be sure.  Histograms are useful for that.

In [ ]:
# Define a list of columns to visualize with box plots
columns_to_visualize = ['Met_Color', 'Automatic', 'Mfg_Guarantee', 'BOVAG_Guarantee',
                      'ABS', 'Airbag_1', 'Airbag_2', 'Airco', 'Automatic_airco',
                      'Boardcomputer', 'CD_Player', 'Central_Lock', 'Powered_Windows',
                      'Power_Steering', 'Radio', 'Mistlamps', 'Sport_Model',
                      'Backseat_Divider', 'Metallic_Rim', 'Radio_cassette', 'Tow_Bar']

for column in columns_to_visualize:
    if column in df.columns:
        plt.figure(figsize=(6, 4))
        ax = sns.countplot(data=df, x=column)
        plt.title(f'Distribution of {column}')
        plt.xlabel(column)
        plt.ylabel('Count')

        # Add counts and percents above the bars
        total = len(df)
        for p in ax.patches:
            height = p.get_height()
            ax.text(p.get_x() + p.get_width()/2.,
                    height + 3,
                    '{:1.0f}\n({:1.1f}%)'.format(height, 100*height/total),
                    ha="center")
        plt.show()
    else:
        print(f"Column '{column}' not found in the DataFrame.")

Yep, they are all 0 and 1 binary indicators. Now let's change the data type to object for all of them.

In [ ]:
# Define a list of columns to convert to object type
columns_to_convert = ['Met_Color', 'Automatic', 'Mfg_Guarantee', 'BOVAG_Guarantee',
                      'ABS', 'Airbag_1', 'Airbag_2', 'Airco', 'Automatic_airco',
                      'Boardcomputer', 'CD_Player', 'Central_Lock', 'Powered_Windows',
                      'Power_Steering', 'Radio', 'Mistlamps', 'Sport_Model',
                      'Backseat_Divider', 'Metallic_Rim', 'Radio_cassette', 'Tow_Bar']

# Iterate through the list and change the data type of each column
for column in columns_to_convert:
    if column in df.columns: # Check if the column exists in the DataFrame
        df[column] = df[column].astype(str)
        print(f"Converted column '{column}' to object type.")
    else:
        print(f"Column '{column}' not found in the DataFrame.")

# Verify the datatype change for the first column in the list (if it exists)
if columns_to_convert and columns_to_convert[0] in df.columns:
    print(f"\nDatatype of '{columns_to_convert[0]}': {df[columns_to_convert[0]].dtypes}")

# Display the head of the first column in the list (if it exists)
if columns_to_convert and columns_to_convert[0] in df.columns:
    print(f"\nHead of '{columns_to_convert[0]}':\n{df[columns_to_convert[0]].head()}")

Now we can confirm the categorical variables.

In [ ]:
# Categorical Variables
# look at the data types of the columns, showing only object type columns
# Use select_dtypes to filter columns by data type
print(df.select_dtypes(include=['object']).head())

These all look good.  

Now let's check our continuous variables.

In [ ]:
# look at the data types of the columns, showing only object type columns
# Use select_dtypes to filter columns by data type
print(df.select_dtypes(include=['int64', 'float64']).head())

Because the month is separate from the year in its own standalone column, it is more categorical in nature than continuous.  Let's fix that.  

In [ ]:
# Change 'Mfg_Month' from int64 to object
df['Mfg_Month'] = df['Mfg_Month'].astype(str)

# Verify the datatype change
print(df['Mfg_Month'].dtypes)

# Display the head of the 'Mfg_Month' column to see the change
print(df['Mfg_Month'].head())

Now let's do a final check of our data types.

In [ ]:
# data types
# look at the data types of the columns
df.dtypes

Looks good!

Now let's practice binning data.  We can bin the weight column into ranges for light, mid-weight, and heavy cars.

In [ ]:
# show a histogram of weight with quartiles to get an idea of bin size

plt.figure(figsize=(10, 6))

# Create the histogram
sns.histplot(data=df, x='Weight', bins=20)

plt.title('Distribution of Car Weight with Quartiles')
plt.xlabel('Weight')
plt.ylabel('Frequency')

# Calculate the quartiles
q1 = df['Weight'].quantile(0.25)
q2 = df['Weight'].quantile(0.50) # Median
q3 = df['Weight'].quantile(0.75)

# Add vertical lines for the quartiles
plt.axvline(q1, color='red', linestyle='--', label=f'Q1 ({q1:.2f})')
plt.axvline(q2, color='green', linestyle='-', label=f'Median ({q2:.2f})')
plt.axvline(q3, color='purple', linestyle='--', label=f'Q3 ({q3:.2f})')

# Add a legend to show the labels for the lines
plt.legend()

plt.show()

In [ ]:
# Bin the weight column into ranges for light, mid-weight, and heavy cars
# Use the quartiles above as cutoffs

# Define the bin edges and labels
bins = [0, 1040, 1085, df['Weight'].max()] # Assuming minimum weight is 0, use max weight for the upper bound
labels = ['light', 'mid-weight', 'heavy']

# Create a new column 'Weight_Bin' by binning the 'Weight' column
df['Weight_Bin'] = pd.cut(df['Weight'], bins=bins, labels=labels, right=True)

# Display the first few rows with the new column
print(df[['Weight', 'Weight_Bin']].head())

# Display the distribution of the new 'Weight_Bin' column
print("\nDistribution of Weight Bins:")
print(df['Weight_Bin'].value_counts())

# Optionally, visualize the distribution of the new bin
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Weight_Bin', order=labels)
plt.title('Count of Cars by Weight Bin')
plt.xlabel('Weight Category')
plt.ylabel('Count')
plt.show()

We can also turn our categorical variables into numeric indicators.  Let's practice that with the Fuel_Type.  

In [ ]:
# Show a histogram of fuel type

plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='Fuel_Type') # Assign the plot to a variable 'ax'

plt.title('Count of Cars by Fuel Type')
plt.xlabel('Fuel Type')
plt.ylabel('Count')

# Add value labels to the bars
total = len(df) # Get the total number of observations for calculating percentages later if needed
for p in ax.patches:
    height = p.get_height() # Get the height of each bar (which represents the count)
    ax.text(p.get_x() + p.get_width()/2., # X-coordinate for the text (center of the bar)
            height + 3, # Y-coordinate for the text (slightly above the top of the bar)
            '{:1.0f}'.format(height), # The text to display (the count)
            ha="center") # Horizontal alignment of the text (center)

plt.show()

There are three fuel types, so we will make two new dummy variables.

In [ ]:
# Make dummy variables for the 'Fuel_Type' column
# 'drop_first=True' drops the first category to avoid multicollinearity (n-1 dummies)

# Create dummy variables, but do NOT drop the original column yet
fuel_type_dummies = pd.get_dummies(df['Fuel_Type'], drop_first=True, dtype=float)

# Rename columns to be more descriptive (optional but recommended)
fuel_type_dummies.columns = [f'Fuel_Type_{col}' for col in fuel_type_dummies.columns]

# Concatenate the original DataFrame with the new dummy variables DataFrame
df = pd.concat([df, fuel_type_dummies], axis=1)

# Display the first few rows to see the original and new dummy columns
print(df.head())

# Display the columns to see the original and new dummy variables
print("\nColumns after creating dummy variables for Fuel_Type:")
df.columns

In this case I kept the original Fuel_Type column so we can still use it for uses that don't require dummy variables (like visualizations). But for modeling we would mostly only use the two new dummy variables and not the original column.

Let's check that the dummy variables makes sense.

In [ ]:
# Show a frequency table of fuel_type_diesel and fuel_type_petrol

print("Frequency Table for Fuel Type:")
print(df[['Fuel_Type_Diesel', 'Fuel_Type_Petrol']].value_counts().sort_index())

Looks good!  We have 17 cars that are CNG, 1264 that are Petrol, and 155 that are Diesel.  This matches our histogram of the original Fuel_Type column.

# Missing Data

Let's check the dataset for missing values.

In [ ]:
# Check for missing values in each column
print("Missing values per column:")
print(df.isnull().sum())

# Check the total number of missing values in the DataFrame
print("\nTotal missing values in the dataset:")
print(df.isnull().sum().sum())

When we have missing data in our target varible (Price) we have to drop those rows.  We can't train a model on an unknown target.

In [ ]:
# Find the index of rows with missing values in the 'Price' column
missing_price_indices = df[df['Price'].isnull()].index

# Drop the rows with missing 'Price' values
df.drop(missing_price_indices, inplace=True)

# Verify that the rows have been dropped by checking for missing 'Price' values again
print("\nMissing values after dropping rows with missing Price:")
print(df.isnull().sum())

# Verify the new shape of the DataFrame
print("\nShape of DataFrame after dropping rows with missing Price:")
df.shape

Age has 23 missing values.  But the age is a function of the manufacturing month and year.  So we can fill in the missing values for these with accurate values.

age_08_04 = (2004 - mfg_year) * 12 + (8 - mfg_month) + 1

In [ ]:
# Identify rows where 'age_08_04' is missing
missing_age_indices = df[df['Age_08_04'].isnull()].index

# Iterate through the identified indices and calculate 'age_08_04'
for index in missing_age_indices:
    mfg_year = df.loc[index, 'Mfg_Year']
    mfg_month = int(df.loc[index, 'Mfg_Month']) # Convert month back to int for calculation

    # Check if both Mfg_Year and Mfg_Month are available for the calculation
    if not pd.isnull(mfg_year) and not pd.isnull(mfg_month):
        calculated_age = (2004 - mfg_year) * 12 + (8 - mfg_month) + 1
        df.loc[index, 'Age_08_04'] = calculated_age

# Convert 'Mfg_Month' back to string type if needed for other parts of the code
df['Mfg_Month'] = df['Mfg_Month'].astype(str)

# Verify that the missing values in 'age_08_04' have been filled
print("\nMissing values after filling missing Age_08_04:")
print(df.isnull().sum())

# Display the rows where Age_08_04 was missing to check the updated values
print("\nRows where Age_08_04 was missing (now filled):")
print(df.loc[missing_age_indices, ['Mfg_Year', 'Mfg_Month', 'Age_08_04']])

Age looks good.  Now let's take a look at KM.  With just 15 values missing, we could drop the rows.  But let's use imputation so you can see how it works.  In this case we are imputing with the column mean.

In [ ]:
# Update the missing values in the KM column using imputation
mean_km = df['KM'].mean() # Calculate the mean
print(f"Imputing missing KM values with the mean: {mean_km:.2f}") # Print the mean used
df['KM'].fillna(mean_km, inplace=True)

# Verify that the missing values in 'KM' have been filled
print("\nMissing values after filling missing KM:")
print(df.isnull().sum())

The last column to address is Quarterly_Tax.  Because more than half the column has missing values, we will drop the column due to poor data quality.  Also, tax is often a function of the car's value so it shouldn't be used to predict price.  Chances are good that if we know the tax, we'd already know the value which is a proxy for price.

In [ ]:
# Drop the 'Quarterly_Tax' column
df.drop('Quarterly_Tax', axis=1, inplace=True)

# Verify that the 'Quarterly_Tax' column has been dropped
print("\nColumns after dropping Quarterly_Tax:")
print(df.columns)

# Verify that there are no missing values in the dropped column (it's not there)
print("\nMissing values after dropping Quarterly_Tax:")
print(df.isnull().sum())

# Verify the new shape of the DataFrame
print("\nShape of DataFrame after dropping Quarterly_Tax:")
df.shape

All missing values have been updated!

# Outliers

In the last module we used the quantile range approach to show us that there is an outlier in the CC column. Note that the Id is 81 but the Python row index is 80.  This is because Python starts counting the first row as zero.

In [ ]:
# find outliers in the CC column

# calculate interquartile range (IQR)
q1 = df['CC'].quantile(0.1)
q3 = df['CC'].quantile(0.9)
iqr = q3 - q1

# define the boundaries for an outlier
lower_bound = q1 - (3 * iqr)
upper_bound = q3 + (3 * iqr)

# find the outliers
outliers = df[(df['CC'] < lower_bound) | (df['CC'] > upper_bound)]

# display the outliers
print(outliers)

Since the CC value of 16000 is likely a typo, let's update it to 1600. This is an example of correcting outliers.

In [ ]:
# Identify the row index of the outlier (the one with CC = 16000)
# We can do this by finding the index where the 'CC' column is 16000
outlier_index = df[df['CC'] == 16000].index

# Check if the index was found and if there is only one such index (as expected)
if not outlier_index.empty and len(outlier_index) == 1:
    # Get the single index
    idx = outlier_index[0]

    # Update the 'CC' value at this specific index to 1600
    df.loc[idx, 'CC'] = 1600
    print(f"Updated the outlier CC value from 16000 to 1600 at index {idx}.")

    # Verify the update by checking the row again
    print("\nRow after updating the CC value:")
    print(df.loc[idx])

else:
    print("Could not find the specific outlier with CC = 16000.")

Now let's use the quantile range approach like we did in the last module to find the remaining outliers.

In [ ]:
## Detecting Outliers in All Numeric Columns

# Select only the numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns

# Iterate through each numeric column to find outliers
for col in numeric_cols:
    print(f"Checking for outliers in column: {col}")

    # calculate interquartile range (IQR)
    q1 = df[col].quantile(0.1)
    q3 = df[col].quantile(0.9)
    iqr = q3 - q1

    # define the boundaries for an outlier
    lower_bound = q1 - (3 * iqr)
    upper_bound = q3 + (3 * iqr)

    # find the outliers
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    # display the outliers for the current column
    if not outliers.empty:
        print(f"Outliers found in {col}: {len(outliers)} outliers")
        # Display only the outlier values from the current column
        print(outliers[col])
    else:
        print(f"No outliers found in {col} using the quantile range method.")
    print("-" * 30) # Separator for clarity between columns

We have outliers remaining in HP, Gears, Weight, and Guarantee_Period.  Let's look at the distribution of those variables wot get a visual sense of how severe the outliers are.

In [ ]:
# Columns for plotting
cols_to_plot = ['HP', 'Gears', 'Weight', 'Guarantee_Period']

# Create subplots for histograms and box plots side by side
fig, axes = plt.subplots(len(cols_to_plot), 2, figsize=(12, 4 * len(cols_to_plot))) # 2 columns for hist and boxplot

for i, col in enumerate(cols_to_plot):
    # Histogram
    sns.histplot(data=df, x=col, ax=axes[i, 0])
    axes[i, 0].set_title(f'Histogram of {col}')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Frequency')

    # Box plot
    sns.boxplot(data=df, x=col, ax=axes[i, 1])
    axes[i, 1].set_title(f'Box Plot of {col}')
    axes[i, 1].set_xlabel(col)

plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()

Looks like there are a handful of cars with HP greater than 120.  Let's look at those rows.

In [ ]:
# Show rows where HP > 120
print("\nRows where HP > 120:")
print(df[df['HP'] > 120])

These high horse power cars all look like they are sport models.  They also tend to be priced higher, so the high HP can be helpful in predicting some of the higher priced cars.  And the values are realistic based on a quick Google search.  So we don't have to do anything with HP outliers.

Next, let's look at Gears.  From the histogram and box plot it looks like most cars have 5 gears.  Let's quantify that to help us make a decision.

In [ ]:
# Count the number of cars where the number of gears is not equal to 5
cars_not_5_gears = df[df['Gears'] != 5].shape[0]

# Calculate the total number of cars
total_cars = df.shape[0]

# Calculate the percentage of cars not equal to 5 gears
percentage_not_5_gears = (cars_not_5_gears / total_cars) * 100

print(f"Number of cars with gears not equal to 5: {cars_not_5_gears}")
print(f"Percentage of cars with gears not equal to 5: {percentage_not_5_gears:.2f}%")

Since almost 97% of the Gears column has the same value, there isn't much variability to take advantage of in modeling.  So we can drop this column.  In this case, looking at potential outliers helped us see that most of the data here is the same.  So this isn't truly an outlier problem to fix, but it is still worth fixing!

In [ ]:
# Drop the Gears column
df.drop('Gears', axis=1, inplace=True)
print("\nColumns after dropping Gears:")
print(df.columns)
print("\nShape of DataFrame after dropping Gears:")
df.shape

We will look at Weight a little later, so now on to Guarantee_Period.  It looks like the vast majority of cars have a 3 month guarantee.  Let's take a closer look at how it relates to price with a scatterplot.  

In [ ]:
# Show a scatterplot of price and guarantee period

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Guarantee_Period', y='Price')
plt.title('Scatter Plot of Price vs. Guarantee Period')
plt.xlabel('Guarantee Period (Months)')
plt.ylabel('Price')
plt.show()

# Print the correlation
correlation = df['Guarantee_Period'].corr(df['Price'])
print(f"The correlation between Guarantee_Period and Price is: {correlation:.4f}")

With such a low correlation to price, I would tend to drop this as a predictor.  Alternately, we could treat this as a categorical variable called something like Extended_Guarantee where a 1 means greater than 3 months and a 0 means 3 months or less.  But for simplicity now, we can just drop the column.

In [ ]:
# Drop the Guarantee_Period column
df.drop('Guarantee_Period', axis=1, inplace=True)
print("\nColumns after dropping Guarantee_Period:")
print(df.columns)
print("\nShape of DataFrame after dropping Guarantee_Period:")
df.shape

Now we've checked all of our outliers except Weight which we will come back to.

# Standardizing Data

Let's practice standardizing data using Z-scores. We will do this on KM.

In [ ]:
# Standardize the 'KM' column using Z-scores
df['KM_Standardized'] = (df['KM'] - df['KM'].mean()) / df['KM'].std()

# Display the head with the original and standardized KM columns
print(df[['KM', 'KM_Standardized']].head())

# Optionally, visualize the distribution of the original and standardized KM
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1) # 1 row, 2 columns, 1st plot
sns.histplot(data=df, x='KM', kde=True)
plt.title('Distribution of Original KM')
plt.xlabel('KM')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2) # 1 row, 2 columns, 2nd plot
sns.histplot(data=df, x='KM_Standardized', kde=True)
plt.title('Distribution of Standardized KM')
plt.xlabel('Standardized KM')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

See how the shape of the distribution doesn't change but the scale does.  The data is now centered around zero with a standard deviation of 1.

Now let's standardize with 0 to 1 scale.

In [ ]:
# Standardize the 'KM' column using Min-Max scaling (0 to 1 scale)
df['KM_Scaled'] = (df['KM'] - df['KM'].min()) / (df['KM'].max() - df['KM'].min())

# Display the head with the original, standardized, and scaled KM columns
print(df[['KM', 'KM_Standardized', 'KM_Scaled']].head())

# Optionally, visualize the distribution of the original, standardized, and scaled KM
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1) # 1 row, 3 columns, 1st plot
sns.histplot(data=df, x='KM', kde=True)
plt.title('Distribution of Original KM')
plt.xlabel('KM')
plt.ylabel('Frequency')

plt.subplot(1, 3, 2) # 1 row, 3 columns, 2nd plot
sns.histplot(data=df, x='KM_Standardized', kde=True)
plt.title('Distribution of Standardized KM (Z-score)')
plt.xlabel('Standardized KM')
plt.ylabel('Frequency')

plt.subplot(1, 3, 3) # 1 row, 3 columns, 3rd plot
sns.histplot(data=df, x='KM_Scaled', kde=True)
plt.title('Distribution of Scaled KM (0-1)')
plt.xlabel('Scaled KM')
plt.ylabel('Frequency')


plt.tight_layout()
plt.show()

Again, the shape of the distribution didn't change but now all the numbers are scaled so that the lowest value in the column is now 0 and the highest value in the column is now 1.  

# Transforming Data

When we have very skewed data it can appear that there are a lot of outliers.  An example of this is the weight column.  Let's look at that distribution.

In [ ]:
# Show a histogram of weight with box plot

plt.figure(figsize=(10, 6))

# Create the histogram
sns.histplot(data=df, x='Weight', bins=20, kde=True) # Added kde=True for density curve

plt.title('Distribution of Car Weight')
plt.xlabel('Weight')
plt.ylabel('Frequency')

plt.show()

# Create the box plot
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='Weight')
plt.title('Box Plot of Car Weight')
plt.xlabel('Weight')
plt.show()

While there aren't a ton of cars beyond the upper whisker, we don't necessarily want to exclude them from our analysis.  Instead we can try transforming the variable.  

In [ ]:
# Transform the weight variable using the johnson su transform

# Initialize the PowerTransformer with the 'yeo-johnson' method
# The 'yeo-johnson' method can handle both positive and negative data
# For weight, which is always positive, 'box-cox' would also be suitable.
# However, 'yeo-johnson' is more general and safer if other variables might have zeros or negatives.
pt = PowerTransformer(method='yeo-johnson', standardize=True) # standardize=True makes the transformed data zero mean, unit variance

# Reshape the data to be a 2D array (required by transform)
weight_reshaped = df['Weight'].values.reshape(-1, 1)

# Fit the transformer and transform the 'Weight' column
df['Weight_JohnsonSU'] = pt.fit_transform(weight_reshaped)

# Display the head with the original and transformed Weight columns
print(df[['Weight', 'Weight_JohnsonSU']].head())

# Optionally, visualize the distribution of the original and transformed Weight
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1) # 1 row, 2 columns, 1st plot
sns.histplot(data=df, x='Weight', kde=True)
plt.title('Distribution of Original Weight')
plt.xlabel('Weight')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2) # 1 row, 2 columns, 2nd plot
sns.histplot(data=df, x='Weight_JohnsonSU', kde=True)
plt.title('Distribution of Weight (Johnson-SU Transform)')
plt.xlabel('Transformed Weight')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.boxplot(data=df, x='Weight')
plt.title('Box Plot of Original Weight')
plt.xlabel('Weight')

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x='Weight_JohnsonSU')
plt.title('Box Plot of Weight (Johnson-SU Transform)')
plt.xlabel('Transformed Weight')

plt.tight_layout()
plt.show()

This isn't a perfect transform, but it is less skewed than the original distribution.  There are now fewer apparent outliers falling outside the whiskers. When the orginal distribution was more skewed, this technique is more effective.

# Data Reduction - Categorical Variables

When categorical variables are converted to dummy variables, they can make a dataset very wide with lots of columns.  This increases the complexity of the dataspace because each new column is another dimension that the models need to account for.  To minimize this, we can combine categories that have similar behavior towards the target variable.  In this case, let's look at how car color impacts price.  

In [ ]:
# Make a table of car colors showing count, mean price, median price, standard deviation of price.
# sort ascending by mean price

# Group by 'Color' and calculate the required statistics
color_price_summary = df.groupby('Color')['Price'].agg(['count', 'mean', 'median', 'std']).reset_index()

# Rename the columns for clarity
color_price_summary.columns = ['Color', 'Count', 'Mean Price', 'Median Price', 'Std Dev Price']

# Sort the table by 'Mean Price' in ascending order
color_price_summary_sorted = color_price_summary.sort_values(by='Mean Price', ascending=True)

# Display the resulting table
print("Summary Table of Price Statistics by Car Color (Sorted by Mean Price):")
color_price_summary_sorted

Based on this we can see that four of the categories have very low counts.  We should combine those with similar categories.  We can combine white, beige, violet with green, and we can combine yellow with grey. That reduces the categories from 10 to 6.  Also, black cars and silver cars have very similar prices so we can combine those.  Green and red are similar too so let's combine those.  That reduces the list to four.

In [ ]:
# Define the mapping for recoding colors
color_mapping = {
    'White': 'Other Color',
    'Beige': 'Other Color',
    'Violet': 'Other Color',
    'Green': 'Other Color',
    'Red': 'Other Color',
    'Black': 'Black/Silver',
    'Silver': 'Black/Silver',
    'Grey': 'Grey/Yellow',
    'Yellow': 'Grey/Yellow',
    'Blue': 'Blue' # Keep Blue as is
}

# Apply the mapping to create the new 'Recoded_Color' column
df['Recoded_Color'] = df['Color'].map(color_mapping).fillna(df['Color']) # Use fillna to keep colors not in the mapping (like 'Blue') as they are

# Display the frequency of the new recoded colors
print("\nFrequency of Recoded Colors:")
print(df['Recoded_Color'].value_counts())

# Verify the recoding by showing the first few rows with original and recoded color
print("\nFirst few rows with original and recoded color:")
print(df[['Color', 'Recoded_Color']].head())

# Make a table of the new recoded car colors showing count, mean price, median price, standard deviation of price.
# sort ascending by mean price

# Group by 'Recoded_Color' and calculate the required statistics
recoded_color_price_summary = df.groupby('Recoded_Color')['Price'].agg(['count', 'mean', 'median', 'std']).reset_index()

# Rename the columns for clarity
recoded_color_price_summary.columns = ['Recoded_Color', 'Count', 'Mean Price', 'Median Price', 'Std Dev Price']

# Sort the table by 'Mean Price' in ascending order
recoded_color_price_summary_sorted = recoded_color_price_summary.sort_values(by='Mean Price', ascending=True)

# Display the resulting table
print("\nSummary Table of Price Statistics by Recoded Car Color (Sorted by Mean Price):")
recoded_color_price_summary_sorted


# Data Reduction - PCA

Principal components analysis (PCA) helps us reduce the number of continuous predictors we have by generating a new smaller set of predictors that have no overlap of information.  

Let's perform PCA on our continuous predictors - Age_08_04, KM, HP, CC, Doors, and Weight.

In [ ]:
# Check correlations for Age_08_04, KM, HP, CC, Doors, and Weight

# Select the columns for PCA
pca_cols = ['Age_08_04', 'KM', 'HP', 'CC', 'Doors', 'Weight']
df_pca = df[pca_cols]

# Calculate the correlation matrix
correlation_matrix = df_pca.corr()

# Display the correlation matrix
print("Correlation Matrix for PCA Variables:")
print(correlation_matrix)

# Optionally visualize the correlation matrix with a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Heatmap of PCA Variables')
plt.show()

These correlations aren't terribly high, but we still might see some advantage to using PCA.

In [ ]:
# Perform PCA on Age_08_04, KM, HP, CC, Doors, and Weight

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Define the features to use for PCA
pca_features = ['Age_08_04', 'KM', 'HP', 'CC', 'Doors', 'Weight']

# Select the features for PCA
df_pca = df[pca_features]

# Standardize the data before applying PCA
# PCA is affected by scale, so you need to scale features before applying it.
scaler = StandardScaler()
df_pca_scaled = scaler.fit_transform(df_pca)

# Apply PCA
# We will choose to keep all components for now to see the variance explained
pca = PCA()
df_pca_transformed = pca.fit_transform(df_pca_scaled)

# Create a DataFrame with the principal components
# The number of components is the number of original features
pca_columns = [f'PC{i+1}' for i in range(df_pca_transformed.shape[1])]
df_pca_components = pd.DataFrame(df_pca_transformed, columns=pca_columns, index=df_pca.index)

# Display the explained variance ratio for each principal component
print("Explained variance ratio by each principal component:")
print(pca.explained_variance_ratio_)

# Display the cumulative explained variance ratio
cumulative_variance_ratio = np.cumsum(pca.explained_variance_ratio_)
print("\nCumulative explained variance ratio:")
print(cumulative_variance_ratio)

# Optionally, visualize the explained variance
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance_ratio) + 1), cumulative_variance_ratio, marker='o', linestyle='--')
plt.title('Explained Variance by Number of Principal Components')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.grid(True)
plt.xticks(range(1, len(cumulative_variance_ratio) + 1))
plt.show()

# Display the PCA loadings (coefficients for each original feature in each component)
# This helps in understanding what each principal component represents
print("\nPCA Loadings (Component direction in original feature space):")
pca_loadings = pd.DataFrame(pca.components_.T, columns=pca_columns, index=pca_features)
pca_loadings

In [ ]:
# Concatenate the original DataFrame with the PCA components DataFrame
# Ensure the indices align correctly when concatenating
df = pd.concat([df, df_pca_components], axis=1)
# Display the head to see the new PCA columns
print(df.head())
# Display the columns to confirm the PCA components are added
print("\nColumns after adding PCA components:")
print(df.columns)
print("\nShape of DataFrame after adding PCA components:")
df.shape

Although we saved all 6 PCs to the dataframe, we won't use all 6 in our modeling.  Based on the output we could try using just 4.